In [ ]:
import os

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever

from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.postprocessor.cohere_rerank import CohereRerank

In [ ]:
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

In [ ]:
documents = SimpleDirectoryReader(
    "../llm-universe/data_base/knowledge_db",
    recursive=True
    ).load_data()

splitter = SentenceSplitter(
    chunk_size=500,
    chunk_overlap=100
)

split_docs = splitter.get_nodes_from_documents(documents)


# 取一部分，进行实验
split_docs = split_docs[:200]


In [ ]:
vector_index = VectorStoreIndex(split_docs)

vector_retriever = vector_index.as_retriever(
    similarity_top_k=8
)

bm25_retriever = BM25Retriever.from_defaults(
    docstore=vector_index.docstore,
    similarity_top_k=8
)

hybrid_retriever = QueryFusionRetriever(
    retrievers=[
        vector_retriever, 
        bm25_retriever
        ],
    similarity_top_k=10,
    num_queries=1,
    mode="reciprocal_rerank",
    use_async=False,
    verbose=True
)

In [ ]:
print("documents:", len(documents))
print("split_docs:", len(split_docs))

In [ ]:
reranker = CohereRerank(
    api_key=os.environ["COHERE_API_KEY"],
    top_n = 5,
)

In [ ]:
query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,
    node_postprocessors=[reranker]
)

In [ ]:
response = query_engine.query('总结一下')
response

In [ ]:
import os

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever

from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.postprocessor.cohere_rerank import CohereRerank


# 1. 配置 LLM 和 Embedding
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")


# 2. 读取文档
documents = SimpleDirectoryReader("./data").load_data()


# 3. 切分文档
splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=80,
)

nodes = splitter.get_nodes_from_documents(documents)


# 4. 建立向量索引
vector_index = VectorStoreIndex(nodes)


# 5. Embedding Retriever
vector_retriever = vector_index.as_retriever(
    similarity_top_k=8
)


# 6. BM25 Retriever
bm25_retriever = BM25Retriever.from_defaults(
    docstore=vector_index.docstore,
    similarity_top_k=8,
)


# 7. 混合检索：融合 BM25 + Embedding
hybrid_retriever = QueryFusionRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    similarity_top_k=10,
    num_queries=1,
    mode="reciprocal_rerank",
    use_async=False,
    verbose=True,
)


# 8. Reranker：从 hybrid 召回的候选里再精排
reranker = CohereRerank(
    api_key=os.environ["COHERE_API_KEY"],
    top_n=5,
)


# 9. Query Engine：Retriever + Reranker + LLM
query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,
    node_postprocessors=[reranker],
)


# 10. 查询
response = query_engine.query("你的问题写在这里")
print(response)